In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

### Reading the table

In [0]:
# Read Silver tables
df_cust = spark.table("dev_project.silver.customer_info") \
               .filter(F.col("is_current") == True)

df_az12 = spark.table("dev_project.silver.erp_cust_az12") \
               .withColumnRenamed("gender", "erp_gender")  # rename before join to avoid ambiguity

df_loc  = spark.table("dev_project.silver.erp_loc_a101")

# Join all three on customer_key
df_dim_customers = df_cust \
    .join(df_az12, on="customer_key", how="left") \
    .join(df_loc,  on="customer_key", how="left") \
    .withColumn(
        "gender",
        F.when(F.col("gender") != "n/a", F.col("gender"))
         .otherwise(F.coalesce(F.col("erp_gender"), F.lit("n/a")))
    ) \
    .select(
        "customer_surrogate_key",
        "customer_id",
        "customer_key",
        "first_name",
        "last_name",
        "marital_status",
        "gender",
        "birthdate",
        "country",
        "created_date"
    )

### Validation

In [0]:
print("\n" + "="*50)
print(" DIM_CUSTOMERS VALIDATION")
print("="*50)
print(f"Total records:      {df_dim_customers.count():,}")
print(f"Null surrogate keys:{df_dim_customers.filter(F.col('customer_surrogate_key').isNull()).count()}")
print(f"Null customer keys: {df_dim_customers.filter(F.col('customer_key').isNull()).count()}")
print(f"Null countries:     {df_dim_customers.filter(F.col('country').isNull()).count()}")
print(f"Null birthdates:    {df_dim_customers.filter(F.col('birthdate').isNull()).count()}")
print(f"Gender breakdown:")
df_dim_customers.groupBy("gender").count().show()
print(f"Country breakdown:")
df_dim_customers.groupBy("country").count().orderBy("count", ascending=False).show()
print("="*50)

Writing into the gold layer

In [0]:
df_dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev_project.gold.dim_customers")

print("dim_customers Gold write complete.")

In [0]:
display(spark.table("dev_project.gold.dim_customers").limit(10))